##### Baixar os dados do Bosla Família os meses de jan/26 a mar/26 Pela API

##### Filtrar os dados pelo município de São Paulo, analisando quanto foi pago em cada mês e o total pago no período.

##### Contar quantos benefícios foram recebidos por cada UF e qual a UF que mais recebeu benefícios no período.

#### Imports

In [ ]:
%pip install requests
%pip install pandas

In [10]:
import requests
import pandas as pd
from zipfile import ZipFile
from io import BytesIO

#### Função para Get

In [6]:
def baixar_e_extrair_dados(url):

    # Envia uma requisição para baixar o arquivo
    response = requests.get(url, timeout=30)

    # Verifica se a requisição foi bem-sucedida
    if response.status_code == 200:

        # Abre o arquivo zip em memória
        with ZipFile(BytesIO(response.content)) as arquivo_zip:
            nomes_arquivos = arquivo_zip.namelist()

            if not nomes_arquivos:
                print("Arquivo ZIP vazio.")
                return None

            # Assume que há apenas um arquivo e extrai
            with arquivo_zip.open(nomes_arquivos[0]) as arquivo_csv:
                df = pd.read_csv(arquivo_csv, sep=';', encoding='ISO-8859-1')
                return df

    else:
        print(f"Falha ao baixar o arquivo. Status code: {response.status_code}")
        return None


#### Filtra os meses

In [8]:
meses_2025_2026 = [
    f"{ano}{mes:02d}"
    for ano in range(2025, 2027)
    for mes in range(1, 13)
]
meses_2025_2026

['202501',
 '202502',
 '202503',
 '202504',
 '202505',
 '202506',
 '202507',
 '202508',
 '202509',
 '202510',
 '202511',
 '202512',
 '202601',
 '202602',
 '202603',
 '202604',
 '202605',
 '202606',
 '202607',
 '202608',
 '202609',
 '202610',
 '202611',
 '202612']

#### Faz chamada na API recursiva e salva os meses

In [11]:
import logging
import time

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

dfs_2025_2026 = []
meses_ok, meses_vazios, meses_erro = [], [], []

total = len(meses_2025_2026)
inicio_total = time.time()

for i, mes in enumerate(meses_2025_2026, start=1):
    url = f"https://portaltransparencia.gov.br/download-de-dados/novo-bolsa-familia/{mes}"
    logger.info(f"[{i}/{total}] Iniciando download | mês={mes} | url={url}")
    inicio = time.time()

    try:
        df_mes = baixar_e_extrair_dados(url)

        if df_mes is not None and not df_mes.empty:
            df_mes["MES"] = mes
            dfs_2025_2026.append(df_mes)
            meses_ok.append(mes)
            logger.info(
                f"[{i}/{total}] Sucesso | mês={mes} | linhas={len(df_mes):,} "
                f"| colunas={df_mes.shape[1]} | tempo={time.time() - inicio:.1f}s"
            )
        else:
            meses_vazios.append(mes)
            logger.warning(f"[{i}/{total}] Sem dados | mês={mes} | tempo={time.time() - inicio:.1f}s")

    except Exception as e:
        meses_erro.append(mes)
        logger.error(f"[{i}/{total}] Erro | mês={mes} | {type(e).__name__}: {e}", exc_info=True)

logger.info(
    f"Download finalizado | ok={len(meses_ok)} | vazios={len(meses_vazios)} "
    f"| erros={len(meses_erro)} | tempo_total={time.time() - inicio_total:.1f}s"
)
if meses_vazios:
    logger.warning(f"Meses sem dados: {meses_vazios}")
if meses_erro:
    logger.error(f"Meses com erro: {meses_erro}")

if dfs_2025_2026:
    df = pd.concat(dfs_2025_2026, ignore_index=True)
    logger.info(f"Concatenação concluída | linhas={len(df):,} | colunas={df.shape[1]}")
    display(df.head())
else:
    logger.critical("Nenhum DataFrame foi carregado. Verifique URLs/função de download.")

2026-10-17 16:55:37 | INFO | [1/24] Iniciando download | mês=202501 | url=https://portaltransparencia.gov.br/download-de-dados/novo-bolsa-familia/202501
2026-10-17 16:56:08 | INFO | [1/24] Sucesso | mês=202501 | linhas=20,500,422 | colunas=10 | tempo=30.8s
2026-10-17 16:56:08 | INFO | [2/24] Iniciando download | mês=202502 | url=https://portaltransparencia.gov.br/download-de-dados/novo-bolsa-familia/202502
2026-10-17 16:56:42 | INFO | [2/24] Sucesso | mês=202502 | linhas=20,473,537 | colunas=10 | tempo=34.3s
2026-10-17 16:56:42 | INFO | [3/24] Iniciando download | mês=202503 | url=https://portaltransparencia.gov.br/download-de-dados/novo-bolsa-familia/202503
2026-10-17 16:57:19 | INFO | [3/24] Sucesso | mês=202503 | linhas=20,431,725 | colunas=10 | tempo=36.2s
2026-10-17 16:57:19 | INFO | [4/24] Iniciando download | mês=202504 | url=https://portaltransparencia.gov.br/download-de-dados/novo-bolsa-familia/202504
2026-10-17 16:57:55 | INFO | [4/24] Sucesso | mês=202504 | linhas=20,363,840

Falha ao baixar o arquivo. Status code: 403


2026-10-17 17:06:53 | WARNING | [22/24] Sem dados | mês=202610 | tempo=0.4s
2026-10-17 17:06:53 | INFO | [23/24] Iniciando download | mês=202611 | url=https://portaltransparencia.gov.br/download-de-dados/novo-bolsa-familia/202611


Falha ao baixar o arquivo. Status code: 403


2026-10-17 17:06:53 | WARNING | [23/24] Sem dados | mês=202611 | tempo=0.4s
2026-10-17 17:06:53 | INFO | [24/24] Iniciando download | mês=202612 | url=https://portaltransparencia.gov.br/download-de-dados/novo-bolsa-familia/202612


Falha ao baixar o arquivo. Status code: 403


2026-10-17 17:06:54 | WARNING | [24/24] Sem dados | mês=202612 | tempo=0.5s
2026-10-17 17:06:54 | INFO | Download finalizado | ok=20 | vazios=4 | erros=0 | tempo_total=676.6s
2026-10-17 17:06:54 | WARNING | Meses sem dados: ['202609', '202610', '202611', '202612']


Falha ao baixar o arquivo. Status code: 403


2026-10-17 17:09:56 | INFO | Concatenação concluída | linhas=387,119,228 | colunas=10


,MÊS COMPETÊNCIA,MÊS REFERÊNCIA,UF,CÓDIGO MUNICÍPIO SIAFI,NOME MUNICÍPIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA,MES
0,202501,202308,SP,7071,SANTOS,***.085.106-**,2.064389e+10,FERNANDA RAMOS TEIXEIRA,"650,00",202501
1,202501,202309,SP,7071,SANTOS,***.085.106-**,2.064389e+10,FERNANDA RAMOS TEIXEIRA,"650,00",202501
2,202501,202310,SP,7071,SANTOS,***.085.106-**,2.064389e+10,FERNANDA RAMOS TEIXEIRA,"650,00",202501
3,202501,202311,SP,7071,SANTOS,***.085.106-**,2.064389e+10,FERNANDA RAMOS TEIXEIRA,"650,00",202501
4,202501,202312,SP,7071,SANTOS,***.085.106-**,2.064389e+10,FERNANDA RAMOS TEIXEIRA,"650,00",202501


#### Printa os meses para conferência do que retornou

In [12]:
# Meses distintos nas colunas de período
for col in ["MÊS COMPETÊNCIA", "MÊS REFERÊNCIA", "MES"]:
    meses = sorted(df[col].dropna().unique())
    print(f"{col}: {len(meses)} meses | min={meses[0]} | max={meses[-1]}")
    print(meses, "\n")

MÊS COMPETÊNCIA: 20 meses | min=202501 | max=202608
[np.int64(202501), np.int64(202502), np.int64(202503), np.int64(202504), np.int64(202505), np.int64(202506), np.int64(202507), np.int64(202508), np.int64(202509), np.int64(202510), np.int64(202511), np.int64(202512), np.int64(202601), np.int64(202602), np.int64(202603), np.int64(202604), np.int64(202605), np.int64(202606), np.int64(202607), np.int64(202608)] 

MÊS REFERÊNCIA: 37 meses | min=202308 | max=202608
[np.int64(202308), np.int64(202309), np.int64(202310), np.int64(202311), np.int64(202312), np.int64(202401), np.int64(202402), np.int64(202403), np.int64(202404), np.int64(202405), np.int64(202406), np.int64(202407), np.int64(202408), np.int64(202409), np.int64(202410), np.int64(202411), np.int64(202412), np.int64(202501), np.int64(202502), np.int64(202503), np.int64(202504), np.int64(202505), np.int64(202506), np.int64(202507), np.int64(202508), np.int64(202509), np.int64(202510), np.int64(202511), np.int64(202512), np.int64(20

#### Colunas necessárias + Conversão do VALOR PARCELA

In [15]:
# df_colunas_necessarias = df[[
#     "MES",
#     "NIS FAVORECIDO",
#     "MÊS COMPETÊNCIA",
#     "UF",
#     "NOME MUNICÍPIO",
#     "VALOR PARCELA"
# ]].copy()

# # "850,00" -> 850.0 (necessário para somar e tirar médias)
# df_colunas_necessarias["VALOR PARCELA"] = (
#     df_colunas_necessarias["VALOR PARCELA"]
#     .astype(str)
#     .str.replace(".", "", regex=False)
#     .str.replace(",", ".", regex=False)
#     .astype(float)
# )

# df_colunas_necessarias.info()
# df_colunas_necessarias.head()

In [18]:
%pip install pyarrow

   ---------------------------------------- 0.0/29.2 MB ? eta -:--:--
   ----------- ---------------------------- 8.4/29.2 MB 57.1 MB/s eta 0:00:01
   -------------------------------------- - 28.3/29.2 MB 80.2 MB/s eta 0:00:01
   ---------------------------------------- 29.2/29.2 MB 75.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gc
import pyarrow as pa
import pyarrow.parquet as pq

cols = ["MES", "NIS FAVORECIDO", "MÊS COMPETÊNCIA", "UF", "NOME MUNICÍPIO", "VALOR PARCELA"]
saida = "bolsa_familia.parquet"
batch = 5_000_000  # reduza para 2_000_000 se faltar memória

writer = None
schema = None

for start in range(0, len(df), batch):
    # iloc + seleção de colunas só materializa a fatia atual
    part = df.iloc[start:start + batch][cols].copy()

    # "1.850,00" -> 1850.0 (float64 evita perda de centavos nas somas)
    part["VALOR PARCELA"] = (
        part["VALOR PARCELA"]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype("float64")
    )

    table = pa.Table.from_pandas(part, preserve_index=False)

    if writer is None:
        schema = table.schema  # fixa o schema no primeiro lote
        writer = pq.ParquetWriter(saida, schema, compression="zstd")
    else:
        table = table.cast(schema)

    writer.write_table(table)

    del part, table
    gc.collect()
    print(f"{min(start + batch, len(df)):,} / {len(df):,}")

writer.close()

# Libera o DF original da memória
del df
gc.collect()


5,000,000 / 387,119,228
10,000,000 / 387,119,228
15,000,000 / 387,119,228
20,000,000 / 387,119,228
25,000,000 / 387,119,228
30,000,000 / 387,119,228
35,000,000 / 387,119,228
40,000,000 / 387,119,228
45,000,000 / 387,119,228
50,000,000 / 387,119,228
55,000,000 / 387,119,228
60,000,000 / 387,119,228
65,000,000 / 387,119,228
70,000,000 / 387,119,228
75,000,000 / 387,119,228
80,000,000 / 387,119,228
85,000,000 / 387,119,228
90,000,000 / 387,119,228
95,000,000 / 387,119,228
100,000,000 / 387,119,228
105,000,000 / 387,119,228
110,000,000 / 387,119,228
115,000,000 / 387,119,228
120,000,000 / 387,119,228
125,000,000 / 387,119,228
130,000,000 / 387,119,228
135,000,000 / 387,119,228
140,000,000 / 387,119,228
145,000,000 / 387,119,228
150,000,000 / 387,119,228
155,000,000 / 387,119,228
160,000,000 / 387,119,228
165,000,000 / 387,119,228
170,000,000 / 387,119,228
175,000,000 / 387,119,228
180,000,000 / 387,119,228
185,000,000 / 387,119,228
190,000,000 / 387,119,228
195,000,000 / 387,119,228
200,00

ArrowKeyError: No type extension with name arrow.py_extension_type found

#### Dashboard Pagamentos Bosla Família

In [36]:
%pip install plotly

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --------------------------------- ------ 8.1/9.7 MB 59.8 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 60.2 MB/s  0:00:00

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- ------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
%pip install duckdb

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   -------- ------------------------------- 2.9/13.2 MB 27.9 MB/s eta 0:00:01
   ---------------------------------------  13.1/13.2 MB 43.2 MB/s eta 0:00:01
   ---------------------------------------- 13.2/13.2 MB 41.4 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import os
from datetime import datetime

import duckdb

PARQUET = "bolsa_familia.parquet"
MES_INICIAL = 202501   # None = usa tudo que existe no arquivo

# Agregação direto no parquet (out-of-core): só o resultado agregado vai para o pandas
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")                # ajuste conforme sua RAM
con.execute("SET temp_directory='duckdb_tmp'")       # spill em disco no COUNT(DISTINCT)
con.execute("SET preserve_insertion_order=false")    # reduz uso de memória

# Detecta o tipo do valor: texto ("850,00") ou numérico
tipos = {r[0]: r[1].upper() for r in
         con.execute(f"DESCRIBE SELECT * FROM read_parquet('{PARQUET}')").fetchall()}
if tipos["VALOR PARCELA"].startswith("VARCHAR"):
    valor_expr = """TRY_CAST(REPLACE(REPLACE("VALOR PARCELA", '.', ''), ',', '.') AS DOUBLE)"""
else:
    valor_expr = 'CAST("VALOR PARCELA" AS DOUBLE)'

filtro = f'WHERE CAST("MÊS COMPETÊNCIA" AS INTEGER) >= {MES_INICIAL}' if MES_INICIAL else ""

agg = con.execute(f"""
    SELECT CAST("MÊS COMPETÊNCIA" AS VARCHAR) AS mes,
           "UF"                               AS uf,
           "NOME MUNICÍPIO"                   AS municipio,
           ROUND(SUM({valor_expr}), 2)        AS valor,
           COUNT(*)                           AS parcelas,
           COUNT(DISTINCT "NIS FAVORECIDO")   AS benef   -- únicos por mês/município
    FROM read_parquet('{PARQUET}')
    {filtro}
    GROUP BY 1, 2, 3
    ORDER BY 1, 2, 3
""").df()
con.close()

dados = agg[["mes", "uf", "municipio", "valor", "parcelas", "benef"]].values.tolist()

TEMPLATE = """<!DOCTYPE html>
<html lang="pt-BR"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Bolsa Família | Painel Executivo</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
:root{
  --navy:#0a1f3d; --blue:#1d4e89; --teal:#0f9d8a; --gold:#d99a1e;
  --ink:#1b2437; --muted:#6b778c; --line:#e3e8f0; --bg:#eef1f6; --card:#fff;
}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);
  font-family:Inter,"Segoe UI",Roboto,Helvetica,Arial,sans-serif;-webkit-font-smoothing:antialiased}

/* Cabeçalho */
header{background:linear-gradient(135deg,var(--navy) 0%,#16467a 100%);color:#fff;padding:28px 40px 56px}
header .tag{font-size:11px;letter-spacing:.18em;text-transform:uppercase;color:#9fc3ee;font-weight:600}
header h1{margin:6px 0 4px;font-size:28px;font-weight:700;letter-spacing:-.01em}
header p{margin:0;color:#c9d9ee;font-size:14px}
.wrap{max-width:1320px;margin:-34px auto 0;padding:0 28px 32px}

/* Filtros */
.filtros{background:var(--card);border-radius:14px;padding:16px 20px;display:flex;flex-wrap:wrap;
  gap:14px;align-items:flex-end;box-shadow:0 8px 24px rgba(10,31,61,.12);border:1px solid var(--line)}
.filtros label,.filtros .lbl{display:flex;flex-direction:column;gap:6px;font-size:11px;font-weight:600;
  text-transform:uppercase;letter-spacing:.06em;color:var(--muted)}
.combo{position:relative}
.combo input{min-width:260px}
.sug{position:absolute;top:100%;left:0;margin:4px 0 0;padding:4px;list-style:none;background:#fff;
  border:1px solid var(--line);border-radius:10px;box-shadow:0 12px 28px rgba(10,31,61,.18);
  min-width:100%;width:280px;max-height:260px;overflow-y:auto;z-index:50;display:none}
.sug li{padding:8px 10px;border-radius:6px;font-size:13px;font-weight:500;cursor:pointer;
  text-transform:none;letter-spacing:0;color:var(--ink)}
.sug li.on,.sug li:hover{background:#eaf1fb;color:var(--blue)}
.sug li.vazio{color:var(--muted);cursor:default;background:none}
.filtros input,.filtros select{height:38px;min-width:170px;padding:0 12px;border:1px solid #cfd7e4;border-radius:8px;
  font-size:14px;color:var(--ink);background:#fff;outline:none;transition:border-color .15s, box-shadow .15s}
.filtros input:focus,.filtros select:focus{border-color:var(--blue);box-shadow:0 0 0 3px rgba(29,78,137,.15)}
.filtros select#uf{min-width:120px}
.acoes{margin-left:auto;display:flex;gap:10px}
button{height:38px;padding:0 20px;border-radius:8px;font-size:14px;font-weight:600;cursor:pointer;transition:all .15s}
.btn-p{background:var(--blue);color:#fff;border:1px solid var(--blue)}
.btn-p:hover{background:var(--navy);box-shadow:0 4px 12px rgba(10,31,61,.3)}
.btn-s{background:#fff;color:var(--blue);border:1px solid #b9c7dc}
.btn-s:hover{background:#f2f6fb}
.status{width:100%;font-size:12px;color:var(--muted);min-height:16px}
.status b{color:var(--ink)} .status .warn{color:#b42318;font-weight:600}

/* KPIs */
.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin:20px 0}
.kpi{background:var(--card);border-radius:14px;padding:18px 20px;border:1px solid var(--line);
  border-left:5px solid var(--c);box-shadow:0 2px 8px rgba(10,31,61,.06);transition:transform .2s, box-shadow .2s}
.kpi:hover{transform:translateY(-3px);box-shadow:0 10px 24px rgba(10,31,61,.12)}
.kpi small{display:block;font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:.08em;color:var(--muted)}
.kpi b{display:block;font-size:28px;margin-top:8px;font-weight:700;letter-spacing:-.02em;color:var(--navy)}
.kpi span{font-size:12px;color:var(--muted)}

/* Blocos de gráficos */
.grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}
.card{background:var(--card);border-radius:14px;border:1px solid var(--line);padding:20px 22px 12px;
  box-shadow:0 2px 8px rgba(10,31,61,.06);transition:box-shadow .2s;animation:fade .5s ease both}
.card:hover{box-shadow:0 10px 26px rgba(10,31,61,.12)}
.card.full{grid-column:1/3}
.card h3{margin:0;font-size:16px;font-weight:700;color:var(--navy)}
.card p{margin:2px 0 8px;font-size:12.5px;color:var(--muted)}
@keyframes fade{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:none}}

.chips{display:flex;flex-wrap:wrap;align-items:center;gap:10px;margin-top:16px;font-size:13px;color:var(--muted)}
.chips b{color:var(--navy);background:#e4edf9;padding:3px 10px;border-radius:999px}
.chips button,.timeline button{height:30px;padding:0 12px;font-size:12px;width:auto}
.timeline{margin:0 0 18px;background:var(--card);border:1px solid var(--line);border-radius:14px;
  padding:12px 18px 10px;box-shadow:0 2px 8px rgba(10,31,61,.06)}
.tl-top{display:flex;flex-wrap:wrap;align-items:center;gap:8px;font-size:12px;color:var(--muted);margin-bottom:6px}
.tl-top b{color:var(--navy);font-size:13px}
.tl-top #tlLbl{margin-right:auto;color:var(--ink);font-weight:600}
.dual{position:relative;height:22px}
.dual .trilho{position:absolute;left:9px;right:9px;top:9px;height:4px;border-radius:2px;background:#dbe3ef}
.dual .faixa{position:absolute;top:9px;height:4px;border-radius:2px;background:var(--blue)}
.dual input[type=range]{position:absolute;left:0;top:0;width:100%;height:22px;margin:0;padding:0;border:0;
  background:none;-webkit-appearance:none;appearance:none;pointer-events:none;outline:none}
.dual input[type=range]::-webkit-slider-runnable-track{background:none;height:22px}
.dual input[type=range]::-moz-range-track{background:none}
.dual input[type=range]::-webkit-slider-thumb{-webkit-appearance:none;box-sizing:border-box;pointer-events:auto;
  width:18px;height:18px;margin-top:2px;border-radius:50%;background:#fff;border:3px solid var(--blue);
  cursor:grab;box-shadow:0 1px 4px rgba(10,31,61,.3)}
.dual input[type=range]::-moz-range-thumb{box-sizing:border-box;pointer-events:auto;width:18px;height:18px;
  border-radius:50%;background:#fff;border:3px solid var(--blue);cursor:grab}
@media(max-width:640px){.timeline{padding:10px 12px}.chips{font-size:12px}}
footer{margin-top:22px;font-size:12px;color:var(--muted);text-align:center;line-height:1.6}
body{overflow-x:hidden}
.card,.kpi{min-width:0}
@media(max-width:900px){.kpis{grid-template-columns:repeat(2,minmax(0,1fr))}.grid{grid-template-columns:minmax(0,1fr)}.card.full{grid-column:auto}.acoes{margin-left:0}}
@media(max-width:640px){
  header{padding:20px 18px 48px}
  header .tag{font-size:10px}
  header h1{font-size:21px;line-height:1.25}
  header p{font-size:13px}
  .wrap{padding:0 12px 24px;margin-top:-30px}
  .filtros{display:grid;grid-template-columns:1fr 1fr;gap:12px;padding:14px}
  .filtros label:first-child,.filtros .combo,.filtros .status,.filtros .acoes{grid-column:1/3}
  .filtros label,.filtros .lbl{min-width:0}
  .filtros input,.filtros select,.filtros select#uf{width:100%;min-width:0;height:42px;font-size:16px}
  .sug{width:100%}
  .acoes{display:grid;grid-template-columns:1fr 1fr;gap:10px}
  button{width:100%;height:42px}
  .kpis{gap:10px;margin:14px 0}
  .kpi{padding:14px;border-left-width:4px}
  .kpi small{font-size:10px}
  .kpi b{font-size:20px;margin-top:6px}
  .kpi span{font-size:11px}
  .grid{gap:12px}
  .card{padding:14px 10px 6px;border-radius:12px}
  .card h3{font-size:15px}
  .card p{font-size:12px}
  footer{font-size:11px}
}
</style></head><body>

<header>
  <div class="tag">Painel Executivo · Programa Social</div>
  <h1>Bolsa Família — Análise de Pagamentos</h1>
  <p>Valores, beneficiários e ticket médio por período, UF e município</p>
</header>

<div class="wrap">
  <form class="filtros" id="form" autocomplete="off">
    <label>UF<select id="uf"></select></label>
    <div class="lbl combo">Município
      <input id="mun" placeholder="Digite para pesquisar…" autocomplete="off">
      <ul class="sug" id="sug"></ul>
    </div>
    <label>Mês inicial<select id="m1"></select></label>
    <label>Mês final<select id="m2"></select></label>
    <div class="acoes">
      <button type="button" class="btn-s" id="limpar">Limpar</button>
      <button type="submit" class="btn-p">Pesquisar</button>
    </div>
    <div class="status" id="status"></div>
  </form>

  <div class="chips" id="chips"></div>

  <section class="kpis">
    <div class="kpi" style="--c:#1d4e89"><small>Valor total pago</small><b id="k1">-</b><span id="k1s"></span></div>
    <div class="kpi" style="--c:#0f9d8a"><small>Beneficiários (média mensal)</small><b id="k2">-</b><span>NIS únicos por mês</span></div>
    <div class="kpi" style="--c:#6b778c"><small>Parcelas pagas</small><b id="k3">-</b><span>Total no período</span></div>
    <div class="kpi" style="--c:#d99a1e"><small>Ticket médio</small><b id="k4">-</b><span>Valor ÷ parcelas</span></div>
  </section>

  <div class="timeline" id="timeline">
    <div class="tl-top">
      <b>Período exibido</b><span id="tlLbl"></span>
      <button type="button" class="btn-s" id="tlUlt">Últimos 6</button>
      <button type="button" class="btn-s" id="tlTodos">Ver tudo</button>
    </div>
    <div class="dual">
      <div class="trilho"></div><div class="faixa" id="faixa"></div>
      <input type="range" id="r1" min="0" max="0" value="0">
      <input type="range" id="r2" min="0" max="0" value="0">
    </div>
  </div>

  <section class="grid">
    <div class="card"><h3>Valor total por mês</h3><p>Clique em um mês para filtrar os demais gráficos</p><div id="g_mes"></div></div>
    <div class="card"><h3>Beneficiários por mês</h3><p>NIS únicos atendidos mensalmente</p><div id="g_benef"></div></div>
    <div class="card"><h3>Ticket médio por mês</h3><p>Valor médio por parcela paga</p><div id="g_ticket"></div></div>
    <div class="card"><h3>Valor total por UF</h3><p>Distribuição regional dos pagamentos</p><div id="g_uf"></div></div>
    <div class="card full"><h3>Top 15 municípios</h3><p>Maiores volumes pagos no recorte selecionado</p><div id="g_mun"></div></div>
  </section>

  <footer>
    Fonte: Portal da Transparência (CGU) · Processamento: Python, Pandas e Plotly<br>
    Beneficiários = NIS únicos por município/mês · Gerado em __GERADO__
  </footer>
</div>

<script>
const D = __DADOS__;   // [mes, uf, municipio, valor, parcelas, benef]
const $ = id => document.getElementById(id);
const C = {blue:'#1d4e89', teal:'#0f9d8a', gold:'#d99a1e', navy:'#0a1f3d', grid:'#e8edf4', txt:'#1b2437'};
const MESES = ['jan','fev','mar','abr','mai','jun','jul','ago','set','out','nov','dez'];
const mesLbl = m => MESES[+m.slice(4)-1] + '/' + m.slice(2,4);
const brl = v => v.toLocaleString('pt-BR',{style:'currency',currency:'BRL'});
const num = v => Math.round(v).toLocaleString('pt-BR');
const compact = v => {
  const a = Math.abs(v), f = (x,s)=> 'R$ ' + x.toLocaleString('pt-BR',{maximumFractionDigits:1}) + s;
  return a>=1e9 ? f(v/1e9,' bi') : a>=1e6 ? f(v/1e6,' mi') : a>=1e3 ? f(v/1e3,' mil') : brl(v);
};
const uniq = a => [...new Set(a)].sort();
const meses = uniq(D.map(r=>r[0]));
const ufs = uniq(D.map(r=>r[1]));

function fillSel(sel, vals){ sel.innerHTML = vals.map(v=>`<option value="${v}">${mesLbl(v)}</option>`).join(''); }
fillSel($('m1'), meses); fillSel($('m2'), meses);

// UF como select
$('uf').innerHTML = '<option value="">Todas</option>' + ufs.map(u=>`<option>${u}</option>`).join('');

// Autocomplete de município
// Obs.: o \\u abaixo é intencional (TEMPLATE não é raw string)
const norm = s => s.normalize('NFD').replace(/[\\u0300-\\u036f]/g,'').toUpperCase();
let munLista = [], sugIdx = -1, sugAtual = [];

function atualizaMunicipios(){
  const uf = $('uf').value;
  munLista = uniq(D.filter(r=>!uf||r[1]===uf).map(r=>r[2]));
}
function abreSug(){
  const q = norm($('mun').value.trim());
  sugAtual = munLista.filter(m=>norm(m).includes(q))
    .sort((a,b)=> (norm(b).startsWith(q)) - (norm(a).startsWith(q)))  // "começa com" primeiro
    .slice(0,50);
  sugIdx = -1;
  $('sug').innerHTML = sugAtual.length
    ? sugAtual.map((m,i)=>`<li data-i="${i}">${m}</li>`).join('')
    : '<li class="vazio">Nenhum município encontrado</li>';
  $('sug').style.display = 'block';
}
function fechaSug(){ $('sug').style.display = 'none'; }
function escolhe(i){ $('mun').value = sugAtual[i]; fechaSug(); }
function destaca(){
  [...$('sug').children].forEach((li,i)=>li.classList.toggle('on', i===sugIdx));
  const on = $('sug').children[sugIdx]; if (on) on.scrollIntoView({block:'nearest'});
}

$('mun').addEventListener('focus', abreSug);
$('mun').addEventListener('input', abreSug);
$('mun').addEventListener('blur', fechaSug);
$('sug').addEventListener('mousedown', e=>{          // mousedown evita perder o foco antes do clique
  e.preventDefault();
  const li = e.target.closest('li[data-i]'); if (li) escolhe(+li.dataset.i);
});
$('mun').addEventListener('keydown', e=>{
  const aberto = $('sug').style.display === 'block';
  if (e.key === 'ArrowDown' && aberto){ e.preventDefault(); sugIdx = Math.min(sugIdx+1, sugAtual.length-1); destaca(); }
  else if (e.key === 'ArrowUp' && aberto){ e.preventDefault(); sugIdx = Math.max(sugIdx-1, 0); destaca(); }
  else if (e.key === 'Enter' && aberto && sugIdx >= 0){ e.preventDefault(); escolhe(sugIdx); }
  else if (e.key === 'Escape'){ fechaSug(); }
});
function resetInputs(){
  $('uf').value=''; $('mun').value='';
  $('m1').value=meses[0]; $('m2').value=meses[meses.length-1];
  atualizaMunicipios();
}

function agrupa(rows, key){
  const m = {};
  rows.forEach(r=>{
    const k = key(r);
    m[k] = m[k] || {valor:0,parc:0,benef:0};
    m[k].valor+=r[3]; m[k].parc+=r[4]; m[k].benef+=r[5];
  });
  return m;
}

const base = {
  paper_bgcolor:'rgba(0,0,0,0)', plot_bgcolor:'rgba(0,0,0,0)',
  font:{family:'Inter, Segoe UI, Arial, sans-serif', size:12, color:C.txt},
  margin:{t:10,l:70,r:20,b:50}, height:340, showlegend:false, dragmode:false,
  hoverlabel:{bgcolor:C.navy, font:{color:'#fff', family:'Inter, Segoe UI, Arial'}, bordercolor:C.navy},
  xaxis:{showgrid:false, linecolor:'#cfd7e4', tickfont:{size:11}},
  yaxis:{gridcolor:C.grid, zeroline:false, tickfont:{size:11}}
};
const L = (extra={}) => {
  const o = Object.assign(JSON.parse(JSON.stringify(base)), extra);
  if (isMob()){
    o.height = extra.height || 270;
    o.font = Object.assign({}, o.font, {size:11});
    o.margin = Object.assign({}, o.margin, {l: Math.min(o.margin.l, 46)});
    o.xaxis = Object.assign({}, o.xaxis, {tickfont:{size: Math.min((o.xaxis.tickfont||{}).size || 11, 10)}});
    o.yaxis = Object.assign({}, o.yaxis, {automargin:true});
  }
  return o;
};
const CFG = {displayModeBar:false, responsive:true};
const isMob = () => window.innerWidth <= 640;
let ultimas = [];
let selMes = null, selUF = null, ksAll = [], win = [0,0], resetWin = true;

function atualizaFaixa(){
  const n = ksAll.length, m = Math.max(1, n-1), a = win[0], b = win[1];
  $('faixa').style.left  = `calc(9px + (100% - 18px) * ${a/m})`;
  $('faixa').style.width = `calc((100% - 18px) * ${(b-a)/m})`;
  $('tlLbl').textContent = n ? `${mesLbl(ksAll[a])} – ${mesLbl(ksAll[b])} · ${b-a+1} de ${n} meses` : '';
}
function configuraSlider(){
  const n = ksAll.length;
  ['r1','r2'].forEach(id=>{ $(id).min = 0; $(id).max = Math.max(0,n-1); $(id).disabled = n<2; });
  $('r1').value = win[0]; $('r2').value = win[1];
  atualizaFaixa();
}
function aplicaJanela(){
  configuraSlider();
  ['g_mes','g_benef','g_ticket'].forEach(id=>Plotly.relayout(id,{'xaxis.range':[win[0]-0.5, win[1]+0.5]}));
}
function onSlide(src){
  let a = +$('r1').value, b = +$('r2').value;
  if (a > b){ if (src==='r1') a = b; else b = a; }
  win = [a,b]; aplicaJanela();
}
$('r1').addEventListener('input', ()=>onSlide('r1'));
$('r2').addEventListener('input', ()=>onSlide('r2'));
$('tlTodos').addEventListener('click', ()=>{ win = [0, Math.max(0,ksAll.length-1)]; aplicaJanela(); });
$('tlUlt').addEventListener('click', ()=>{ const n = ksAll.length; win = [Math.max(0,n-JANELA), Math.max(0,n-1)]; aplicaJanela(); });

// Filtro por clique (cross-filter): clicar de novo na mesma seleção remove o filtro
function ligaEventos(){
  const liga = (id, fn)=>{ const el = $(id); if (!el.on) return; el.removeAllListeners('plotly_click'); el.on('plotly_click', fn); };
  const porMes = ev=>{ const k = ksAll[ev.points[0].pointNumber]; if (!k) return; selMes = (selMes===k) ? null : k; render(ultimas); };
  ['g_mes','g_benef','g_ticket'].forEach(id=>liga(id, porMes));
  liga('g_uf', ev=>{ const u = ev.points[0].x; selUF = (selUF===u) ? null : u; render(ultimas); });
}
$('chips').addEventListener('click', e=>{
  if (e.target.id === 'limpaSel'){ selMes = null; selUF = null; render(ultimas); }
});
const merge = (a,b) => Object.assign({}, a, b);

// Eixo mensal com barra de rolagem: abre nos últimos JANELA meses
const JANELA = 6;
function eixoMes(n){
  // janela controlada pela barra única (win); eixo categórico usa índices
  return merge(base.xaxis, {
    type:'category', tickangle:'auto', automargin:true, fixedrange:true,
    range:[win[0]-0.5, win[1]+0.5]
  });
}

function pesquisar(){
  const uf = $('uf').value, mun = norm($('mun').value.trim());
  let a = $('m1').value, b = $('m2').value, aviso = '';
  if (a > b) { [a,b] = [b,a]; $('m1').value=a; $('m2').value=b; }

  const ufOk = !uf || ufs.includes(uf);
  const munOk = !mun || D.some(r=>norm(r[2])===mun && (!uf||r[1]===uf));
  if (!ufOk) aviso = `UF "${uf}" não encontrada.`;
  else if (!munOk) aviso = `Município "${mun}" não encontrado${uf?' em '+uf:''}.`;

  const rows = (aviso) ? [] : D.filter(r=>(!uf||r[1]===uf)&&(!mun||norm(r[2])===mun)&&r[0]>=a&&r[0]<=b);
  $('status').innerHTML = aviso ? `<span class="warn">${aviso}</span>` :
    `Exibindo: <b>${uf||'Brasil'}</b> · <b>${$('mun').value.trim()||'todos os municípios'}</b> · <b>${mesLbl(a)}</b> a <b>${mesLbl(b)}</b>`;
  resetWin = true; selMes = null; selUF = null;
  render(rows);
}

function render(base0){
  ultimas = base0;
  const mob = isMob();
  ksAll = uniq(base0.map(r=>r[0]));
  if (selMes && !ksAll.includes(selMes)) selMes = null;
  // Cada visual ignora o próprio filtro (destaca em vez de filtrar), como no Power BI
  const rowsMes = base0.filter(r=>!selUF  || r[1]===selUF);                         // gráficos mensais
  const rowsUF  = base0.filter(r=>!selMes || r[0]===selMes);                        // gráfico de UF
  const rows    = base0.filter(r=>(!selUF||r[1]===selUF) && (!selMes||r[0]===selMes)); // KPIs e Top 15
  const nk = Math.max(0, ksAll.length-1);
  if (resetWin){ win = [Math.max(0,ksAll.length-JANELA), nk]; resetWin = false; }
  win = [Math.min(win[0],nk), Math.min(win[1],nk)];
  configuraSlider();
  const hGraf = mob ? 270 : 320;
  const short = k => { if (!mob) return k; const [n,u] = k.split(' / '); return (n.length>13 ? n.slice(0,12)+'…' : n) + ' / ' + u; };
  const nMeses = uniq(rows.map(r=>r[0])).length || 1;
  const valor = rows.reduce((s,r)=>s+r[3],0);
  const parc  = rows.reduce((s,r)=>s+r[4],0);
  const benef = rows.reduce((s,r)=>s+r[5],0);
  $('k1').textContent = compact(valor);
  $('k1s').textContent = brl(valor);
  $('k2').textContent = num(benef/nMeses);
  $('k3').textContent = num(parc);
  $('k4').textContent = parc ? brl(valor/parc) : '-';

  // Mês
  const pm = agrupa(rowsMes, r=>r[0]), ks = ksAll, lb = ks.map(mesLbl);
  ks.forEach(k=>{ pm[k] = pm[k] || {valor:0,parc:0,benef:0}; });
  Plotly.react('g_mes',[{x:lb,y:ks.map(k=>pm[k].valor),type:'bar',marker:{color:ks.map(k=>(!selMes||k===selMes)?C.blue:'#b9c8de')},
    text:ks.map(k=>compact(pm[k].valor)),textposition:mob?'none':'outside',cliponaxis:false,textfont:{size:11},
    hovertemplate:'<b>%{x}</b><br>Valor: %{text}<extra></extra>'}],
    L({yaxis:merge(base.yaxis,{tickformat:'~s'}), xaxis:eixoMes(ks.length), height:hGraf, margin:{t:30,l:55,r:20,b:50}}), CFG);

  Plotly.react('g_benef',[{x:lb,y:ks.map(k=>pm[k].benef),type:'scatter',mode:'lines+markers',
    line:{color:C.teal,width:3,shape:'spline'},marker:{size:ks.map(k=>k===selMes?14:8),color:ks.map(k=>k===selMes?C.teal:'#fff'),line:{color:C.teal,width:2.5}},
    fill:'tozeroy',fillcolor:'rgba(15,157,138,.12)',
    hovertemplate:'<b>%{x}</b><br>Beneficiários: %{y:,.0f}<extra></extra>'}],
    L({xaxis:eixoMes(ks.length), yaxis:merge(base.yaxis,{tickformat:'~s',rangemode:'tozero'}), height:hGraf, margin:{t:30,l:55,r:20,b:50}}), CFG);

  Plotly.react('g_ticket',[{x:lb,y:ks.map(k=>pm[k].parc?pm[k].valor/pm[k].parc:0),type:'scatter',mode:'lines+markers+text',
    line:{color:C.gold,width:3,shape:'spline'},marker:{size:ks.map(k=>k===selMes?14:8),color:ks.map(k=>k===selMes?C.gold:'#fff'),line:{color:C.gold,width:2.5}},
    text:ks.map(k=>pm[k].parc?brl(pm[k].valor/pm[k].parc):''),textposition:mob?'none':'top center',textfont:{size:11},
    hovertemplate:'<b>%{x}</b><br>Ticket médio: %{y:,.2f}<extra></extra>'}],
    L({xaxis:eixoMes(ks.length), yaxis:merge(base.yaxis,{tickprefix:'R$ '}), height:hGraf, margin:{t:30,l:70,r:30,b:50}}), CFG);

  // UF
  const pu = agrupa(rowsUF, r=>r[1]), ku = Object.keys(pu).sort((x,y)=>pu[y].valor-pu[x].valor);
  Plotly.react('g_uf',[{x:ku,y:ku.map(k=>pu[k].valor),type:'bar',
    marker:{color:ku.map((k,i)=>selUF ? (k===selUF?C.navy:'#b9c8de') : (i===0?C.navy:C.blue)),opacity:.92},
    customdata:ku.map(k=>[compact(pu[k].valor), pu[k].parc?brl(pu[k].valor/pu[k].parc):'-']),
    hovertemplate:'<b>%{x}</b><br>Valor: %{customdata[0]}<br>Ticket médio: %{customdata[1]}<extra></extra>'}],
    L({yaxis:merge(base.yaxis,{tickformat:'~s'}), xaxis:merge(base.xaxis,{type:'category',tickmode:'array',tickvals:ku,ticktext:ku,tickangle:mob?-90:'auto',tickfont:{size:mob?9:11}}), margin:{t:30,l:55,r:20,b:mob?44:50}}), CFG);

  // Top 15 municípios
  const pn = agrupa(rows, r=>r[2]+' / '+r[1]);
  const kn = Object.keys(pn).sort((x,y)=>pn[y].valor-pn[x].valor).slice(0,15).reverse();
  Plotly.react('g_mun',[{y:kn,x:kn.map(k=>pn[k].valor),type:'bar',orientation:'h',
    marker:{color:kn.map((k,i)=>i===kn.length-1?C.navy:C.teal)},
    text:kn.map(k=>compact(pn[k].valor)),textposition:'outside',cliponaxis:false,textfont:{size:mob?10:11},
    hovertemplate:'<b>%{y}</b><br>Valor: %{text}<extra></extra>'}],
    L({height:mob?480:520, margin:{t:10,l:mob?10:210,r:mob?68:90,b:40},
       xaxis:merge(base.xaxis,{showgrid:true,gridcolor:C.grid,tickformat:'~s',nticks:mob?4:10}),
       yaxis:merge(base.yaxis,{gridcolor:'rgba(0,0,0,0)',automargin:true,tickmode:'array',tickvals:kn,ticktext:kn.map(short),tickfont:{size:mob?10:11}})}), CFG);

  const sel = [selMes && `<b>${mesLbl(selMes)}</b>`, selUF && `<b>${selUF}</b>`].filter(Boolean);
  $('chips').innerHTML = sel.length
    ? `Filtro por clique: ${sel.join(' ')} <button type="button" class="btn-s" id="limpaSel">Limpar seleção</button>`
    : 'Dica: clique em um mês ou em uma UF para filtrar os demais gráficos.';
  setTimeout(ligaEventos, 0);
}

$('uf').addEventListener('change', ()=>{ $('mun').value=''; atualizaMunicipios(); });
$('form').addEventListener('submit', e=>{e.preventDefault(); pesquisar();});
$('limpar').addEventListener('click', ()=>{resetInputs(); pesquisar();});

// Redesenha ao girar o aparelho / cruzar o breakpoint mobile
let wasMob = isMob(), tmr;
window.addEventListener('resize', ()=>{
  clearTimeout(tmr);
  tmr = setTimeout(()=>{ if (isMob() !== wasMob){ wasMob = isMob(); render(ultimas); } }, 200);
});
resetInputs(); pesquisar();
</script></body></html>"""


html = (
    TEMPLATE
    .replace("__DADOS__", json.dumps(dados, ensure_ascii=False))
    .replace("__GERADO__", datetime.now().strftime("%d/%m/%Y"))
)

os.makedirs("docs", exist_ok=True)
with open("docs/index.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"OK: docs/index.html ({len(dados):,} linhas agregadas)")

OK: docs/index.html (111,411 linhas agregadas)
